# How is CT1 related to T1? Three linear views, looked at directly

A data-consistency operator `E: CT1 -> T1` that is **linear in CT1** needs a prior on whatever it
is parameterised by. This notebook draws that object three ways, on the same validation slices, so
each can be judged by eye and by a few numbers:

| section | object | the model it would support |
|---|---|---|
| 1 | difference `D = CT1 - T1` | additive: `CT1 = T1 + D`, prior on `D` (sparse? heavy-tailed?) |
| 2 | quotient `Q = CT1 / T1` | multiplicative: `T1 = CT1 / Q` -- the `gain` operator's map `a = 1/Q` |
| 3 | transfer function | `T1 = sum_j a_j * (B_j CT1)` -- the `kfilter` operator, global or per pixel |

Nothing here is trained. The fits in section 3 use T1 itself, so they are **oracles**: they show
what a perfectly predicted gain map would look like, not what a network can predict from T2/FLAIR.

`draw()` resamples the slices; every later cell reads the globals it sets.

## Knobs

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
%matplotlib inline

if os.path.basename(os.getcwd()) == "notebooks":     # safe to re-run: only steps up once
    os.chdir("..")
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import datasets                                       # noqa: F401  (registers the loaders)
from datasets.registry import build_loader
from models.forward_ops import radial_bands
from visualization import plot_image, subplot_images, plot_hist, subplot_hists
from visualization.image import set_display_orient
set_display_orient("radiological")                    # DISPLAY ONLY; k-space panels opt out

# ---- data: the same slices and scaling the forward-operator ladder trains on -----------------
ROOT = "../datasets/NYUMets_h5_val"
IMAGE_KEY = "img_raw"
SCALES = [2800.0, 2500.0, 2390.0, 2370.0]             # FLAIR, T1, CT1, T2
SLICE_RANGE = (40, 110)                               # None = every slice
N_SLICES = 256                                        # random val slices to pool
SEED = 0
BATCH, NUM_WORKERS = 16, 8
MIN_BRAIN = 0.05                                      # drop slices with less mask than this
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- display ---------------------------------------------------------------------------------
N_SHOW = 4                                            # rows in the image panels
SHOW = None                                           # explicit slice positions, or None = first N_SHOW

# ---- analysis --------------------------------------------------------------------------------
ENH_Q = 0.98          # per slice, the top (1 - ENH_Q) of |residual| is treated as "enhancing"
FLOOR_FRAC = 0.25     # quotient is only formed where T1 > FLOOR_FRAC * median brain T1
SIGMAS = (1, 2, 4, 8, 16)   # px: smoothing scales for the smooth + sparse splits and local fits
NRAD = 64             # radial bins for spectra
BANDS = 8             # kfilter bands for the GLOBAL transfer function
J_LOCAL = 4           # kfilter bands for the per-pixel gain maps (J*J products are held in memory)
LOCAL_SIGMA = 4       # px: window of the local least-squares fit that is DISPLAYED
N_LOCAL = 64          # slices used for the local fits
RIDGE = 1.0e-3        # local fit ridge, pulling gains toward 1 where a band has no energy

## Load

In [ ]:
cfg = dict(name="nyumets_guided", root=ROOT, x0_idx=2, x1_idx=1, cond_idx=[0, 3],
           image_key=IMAGE_KEY, scales=SCALES, guide_mode="none", deterministic=True,
           random_flips=False, crop_size=None, center_crop=None,
           batch_size=BATCH, num_workers=NUM_WORKERS)
if SLICE_RANGE is not None:
    cfg["slice_range"] = list(SLICE_RANGE)
full = build_loader(cfg, shuffle=False, drop_last=False)


def draw(n=N_SLICES, seed=SEED):
    """Pool n random val slices into the globals T1, CT1, FLAIR, T2 (N, H, W) and M (bool)."""
    global T1, CT1, FLAIR, T2, M, N, IDX
    ds = full.dataset
    idx = np.random.default_rng(seed).choice(len(ds), size=min(n, len(ds)), replace=False)
    loader = DataLoader(Subset(ds, idx.tolist()), batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS)
    out = [[], [], [], []]
    for batch in loader:
        for acc, t in zip(out, batch[:4]):
            acc.append(t)
    x0, x1, cond, m = (torch.cat(a).float() for a in out)
    m = m[:, 0] > 0
    keep = m.float().mean((1, 2)) >= MIN_BRAIN
    CT1, T1 = x0[keep, 0].to(DEVICE), x1[keep, 0].to(DEVICE)
    FLAIR, T2 = cond[keep, 0].to(DEVICE), cond[keep, 1].to(DEVICE)
    M, IDX = m[keep].to(DEVICE), idx[keep.numpy()]
    N = int(keep.sum())
    print("%d slices of %d kept (>= %.0f%% mask), frame %s, device %s"
          % (N, len(idx), 100 * MIN_BRAIN, tuple(T1.shape[-2:]), DEVICE))


draw()

## Helpers

In [ ]:
def rows():
    """Slice positions shown in the image panels."""
    return list(range(min(N_SHOW, N))) if SHOW is None else list(SHOW)


def brain(v, m=None):
    """Values inside the mask as a flat float64 numpy array."""
    return v[M if m is None else m].double().cpu().numpy()


def pct(v, q, m=None):
    return float(np.percentile(brain(v, m), q))


def blur(x, sigma):
    """Gaussian blur over the last two dims, reflect-padded. x: (N, H, W) or (N, C, H, W)."""
    if sigma <= 0:
        return x
    shape = x.shape
    x = x.reshape(-1, 1, *shape[-2:])
    r = int(min(np.ceil(3 * sigma), min(shape[-2:]) - 1))
    k = torch.exp(-0.5 * (torch.arange(-r, r + 1, device=x.device, dtype=x.dtype) / sigma) ** 2)
    k = k / k.sum()
    x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="reflect"), k.view(1, 1, 1, -1))
    x = F.conv2d(F.pad(x, (0, 0, r, r), mode="reflect"), k.view(1, 1, -1, 1))
    return x.reshape(shape)


def mblur(f, m, sigma):
    """Blur that only averages over mask pixels (normalised convolution)."""
    w = m.to(f.dtype)
    return blur(f * w, sigma) / blur(w, sigma).clamp_min(1e-6)


def affine_fit(x, y, m, trim=ENH_Q):
    """Per slice, y ~ g * x + c over the mask, refitted once without the top (1 - trim) of
    |residual| so a few enhancing pixels do not set the gain. Returns g, c, trimmed-out mask."""
    def fit(w):
        n = w.sum((1, 2)).clamp_min(1)
        mx, my = (w * x).sum((1, 2)) / n, (w * y).sum((1, 2)) / n
        dx = x - mx[:, None, None]
        g = (w * dx * (y - my[:, None, None])).sum((1, 2)) / (w * dx * dx).sum((1, 2)).clamp_min(1e-12)
        return g, my - g * mx

    def resid(g, c):
        return y - (g[:, None, None] * x + c[:, None, None])

    g, c = fit(m.to(x.dtype))
    r = resid(g, c).abs()
    thr = torch.stack([torch.quantile(r[i][m[i]], trim) for i in range(len(x))])
    out = m & (r > thr[:, None, None])
    g, c = fit((m & ~out).to(x.dtype))
    return g, c, out


def sparsity(v):
    """How concentrated is the energy of v (about its median)?  top1 / top5 = share of sum v^2
    held by the largest 1% / 5% of entries; l1/l2 = ||v||_1 / (sqrt(n) ||v||_2), 1 = flat."""
    v = np.asarray(v, np.float64).ravel()
    v = v - np.median(v)
    e = np.sort(v * v)[::-1]
    cum = np.cumsum(e) / max(e.sum(), 1e-300)
    n = len(v)
    return dict(std=v.std(), kurtosis=(v ** 4).mean() / max((v * v).mean() ** 2, 1e-300),
                top1=cum[max(int(0.01 * n) - 1, 0)], top5=cum[max(int(0.05 * n) - 1, 0)],
                l1l2=np.abs(v).sum() / max(np.sqrt(n * (v * v).sum()), 1e-300))


_rng = np.random.default_rng(0)
REFS = {"Gaussian (reference)": sparsity(_rng.standard_normal(400000)),
        "Laplace (reference)": sparsity(_rng.laplace(size=400000))}


def table(rows_, cols=None, fmt="%10.4g", title=None):
    cols = cols or list(next(iter(rows_.values())))
    w = max(len(k) for k in rows_) + 2
    if title:
        print(title)
    print(" " * w + "".join("%10s" % c for c in cols))
    for k, r in rows_.items():
        print(k.ljust(w) + "".join(fmt % r[c] for c in cols))


def concentration(series, title):
    """Share of sum v^2 held by the largest fraction f of entries, for each series."""
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for label, v in series.items():
        v = np.asarray(v, np.float64).ravel()
        v = v - np.median(v)
        e = np.sort(v * v)[::-1]
        f = np.arange(1, len(e) + 1) / len(e)
        k = np.unique(np.geomspace(1, len(e), 400).astype(int)) - 1
        ax.semilogx(f[k], (np.cumsum(e) / e.sum())[k], label=label)
    ax.set(xlabel="fraction of brain pixels (largest first)", ylabel="share of energy",
           title=title, ylim=(0, 1.02))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()

## 0. The slices

Each contrast on its own window (they are different contrasts, so a shared window would black out
all but the brightest).

In [ ]:
R = rows()
subplot_images([[T1[i], CT1[i], FLAIR[i], T2[i]] for i in R],
               col_titles=["T1", "CT1", "FLAIR", "T2"], share_window=False, p=(1, 99.5),
               row_labels=["#%d" % i for i in R], suptitle="validation slices (scaled raw)")

## 1. Difference `D = CT1 - T1`

Two versions. The raw difference contains whatever whole-image gain and offset separate the two
acquisitions; `D_fit` removes a per-slice `CT1 ~ g * T1 + c` first (fitted without the top
`1 - ENH_Q` of residuals). If `g` and `c` scatter widely, a prior on `D` has to be stated on
`D_fit`, and the operator needs a per-scan gain.

**Look for:** is `D_fit` near zero almost everywhere with a few large values (sparse), and is its
tail heavier than the Laplace reference? Does the gradient of `D` concentrate more than `D` itself?

In [ ]:
D = CT1 - T1
g_f, c_f, ENH = affine_fit(T1, CT1, M)
D_fit = CT1 - (g_f[:, None, None] * T1 + c_f[:, None, None])

subplot_hists({"gain g": g_f.cpu().numpy(), "offset c": c_f.cpu().numpy()}, bins=40, legend=False,
              xlabel="per-slice value", suptitle="per-slice fit  CT1 ~ g * T1 + c  (enhancement trimmed)")
print("g: median %.3f, 10-90%% [%.3f, %.3f]   c: median %.4f, 10-90%% [%.4f, %.4f]" % (
    g_f.median(), g_f.quantile(0.1), g_f.quantile(0.9),
    c_f.median(), c_f.quantile(0.1), c_f.quantile(0.9)))

In [ ]:
V_IMG = max(pct(T1, 99.5), pct(CT1, 99.5))
V_D = float(np.percentile(np.abs(brain(D)), 99.5))
subplot_images([[T1[i], CT1[i], ENH[i].float(), D[i] * M[i], D_fit[i] * M[i]] for i in R],
               col_titles=["T1", "CT1", "top %.0f%% |D_fit|" % (100 * (1 - ENH_Q)), "D", "D_fit"],
               cmap=["gray", "gray", "gray", "bwr", "bwr"],
               vmin=[0.0, 0.0, 0.0, -V_D, -V_D], vmax=[V_IMG, V_IMG, 1.0, V_D, V_D],
               row_labels=["#%d" % i for i in R], colorbar=True, cbar_label="difference",
               panel_size=(2.8, 2.9),
               suptitle="D = CT1 - T1 and D_fit (per-slice gain/offset removed), "
                        "one fixed window +-%.3f" % V_D)

In [ ]:
d, df = brain(D), brain(D_fit)
ref = np.random.default_rng(1)
plot_hist({"D = CT1 - T1": d, "D_fit": df,
           "Gaussian, std of D_fit": ref.standard_normal(len(df)) * df.std(),
           "Laplace, mean |D_fit|": ref.laplace(size=len(df)) * np.abs(df - np.median(df)).mean()},
          bins=200, symmetric=True, p=99.95, log=True, vlines=0.0,
          xlabel="difference (scaled units), brain pixels", title="is the difference heavy-tailed?",
          figsize=(7, 4.2))

gy = lambda v: v[:, 1:, :] - v[:, :-1, :]                 # vertical finite difference
mg = M[:, 1:, :] & M[:, :-1, :]
stats = {"D": sparsity(d), "D_fit": sparsity(df),
         "grad D_fit": sparsity(brain(gy(D_fit), mg)),
         "T1 (for scale)": sparsity(brain(T1)), "grad T1": sparsity(brain(gy(T1), mg)), **REFS}
table(stats, title="\nenergy concentration inside the brain (about the median)")
concentration({"D": d, "D_fit": df, "grad D_fit": brain(gy(D_fit), mg),
               "Gaussian": ref.standard_normal(200000), "Laplace": ref.laplace(size=200000)},
              "difference: how few pixels hold the energy?")

## 2. Quotient `Q = CT1 / T1`

Only formed where `T1 > FLOOR_FRAC * median(T1)`: dividing by dark pixels (CSF, bone, air) turns
noise into arbitrarily large ratios, which is the main practical argument against a quotient
prior. `log Q` is the natural variable -- a gain of 2 and of 1/2 are equally far from 1.

The joint histogram is the direct test of *additive vs multiplicative*: normal tissue lying on a
line **through the origin** with slope `g` is a gain; a line **parallel to the diagonal** is an
offset. The coloured curves are quantiles of CT1 at each T1; enhancement is whatever lifts the
upper quantiles off the median.

**Look for:** is `log Q` a smooth field plus a few sharp spots? The split at several scales below
puts numbers on that.

In [ ]:
FLOOR = FLOOR_FRAC * float(np.median(brain(T1)))
VALID = M & (T1 > FLOOR)
Q = torch.where(VALID, CT1 / T1.clamp_min(FLOOR), torch.ones_like(T1))
LQ = torch.log(Q.clamp_min(1e-3))
print("quotient formed on %.1f%% of mask pixels (T1 > %.4f)"
      % (100 * float(VALID.sum()) / float(M.sum()), FLOOR))

S_SHOW = SIGMAS[len(SIGMAS) // 2 + 1] if len(SIGMAS) > 2 else SIGMAS[-1]
LQ_smooth = torch.where(VALID, mblur(LQ, VALID, S_SHOW), torch.zeros_like(LQ))
LQ_rem = (LQ - LQ_smooth) * VALID
V_L = float(np.percentile(np.abs(brain(LQ, VALID)), 99.5))
subplot_images([[T1[i], CT1[i], LQ[i], LQ_smooth[i], LQ_rem[i]] for i in R],
               col_titles=["T1", "CT1", "log Q", "smooth, sigma=%d" % S_SHOW, "remainder"],
               panel_size=(2.8, 2.9),
               cmap=["gray", "gray", "bwr", "bwr", "bwr"],
               vmin=[0.0, 0.0, -V_L, -V_L, -V_L], vmax=[V_IMG, V_IMG, V_L, V_L, V_L],
               row_labels=["#%d" % i for i in R], colorbar=True, cbar_label="log Q",
               suptitle="log quotient = smooth field + remainder (white = Q of 1, window +-%.2f)" % V_L)

subplot_hists({"Q = CT1 / T1": brain(Q, VALID), "log Q": brain(LQ, VALID)}, bins=200, p=99.9,
              log=True, vlines=[{"Q = 1": 1.0}, {"log Q = 0": 0.0}], xlabel=["Q", "log Q"],
              suptitle="quotient inside the brain (T1 above the floor)")

In [ ]:
def joint_hist(a, b, lo, hi, bins=240):
    """counts[y, x] of (a on x, b on y), plus the bin centres."""
    Hc, _, _ = np.histogram2d(b, a, bins=bins, range=[[lo, hi], [lo, hi]])
    return Hc, lo + (np.arange(bins) + 0.5) * (hi - lo) / bins


def show_joint(a, b, xlabel, ylabel, title, lines=()):
    hi = max(np.percentile(a, 99.9), np.percentile(b, 99.9))
    Hc, ctr = joint_hist(a, b, 0.0, hi)
    nb = len(ctr)
    px = lambda v: np.asarray(v) / hi * nb - 0.5                     # value -> pixel column
    py = lambda v: nb - 1 - px(v)                                    # rows are flipped: up = larger
    fig, ax = plt.subplots(figsize=(6, 6))
    plot_image(np.log10(Hc[::-1] + 1.0), cmap="magma", ax=ax, axis_off=False, orient=None, show=False)
    ok = Hc.sum(0) > 200                                             # columns with enough pixels
    cdf = np.cumsum(Hc, 0) / np.maximum(Hc.sum(0), 1)
    for q, col in ((0.5, "cyan"), (0.9, "lime"), (0.99, "yellow")):
        val = ctr[np.argmax(cdf >= q, 0)]
        ax.plot(px(ctr[ok]), py(val[ok]), color=col, lw=1.2, label="p%g of %s" % (100 * q, ylabel))
    ax.plot(px([0, hi]), py([0, hi]), "w--", lw=0.8, label="identity")
    for label, gg, cc in lines:
        ax.plot(px([0, hi]), py([cc, gg * hi + cc]), "w:", lw=1.2, label=label)
    t = np.linspace(0, hi, 6)
    ax.set_xticks(px(t)); ax.set_xticklabels(["%.2f" % v for v in t])
    ax.set_yticks(py(t)); ax.set_yticklabels(["%.2f" % v for v in t])
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title, xlim=(-0.5, nb - 0.5), ylim=(nb - 0.5, -0.5))
    ax.legend(fontsize=7, loc="lower right")
    plt.show()


show_joint(brain(T1), brain(CT1), "T1", "CT1", "joint histogram, log10 counts (brain)",
           lines=[("median fit: g=%.2f, c=%.3f" % (g_f.median(), c_f.median()),
                   float(g_f.median()), float(c_f.median()))])

In [ ]:
# Smooth + sparse split at several scales, for the additive and the multiplicative variable.
# `smooth std` is how much a slowly varying field explains; top1 / kurtosis describe what is left.
# The representation whose remainder is MORE concentrated at a LARGER sigma has the simpler prior.
split = {}
for s in SIGMAS:
    for name, v, m in (("D_fit", D_fit, M), ("log Q", LQ, VALID)):
        sm = mblur(v, m, s)
        st = sparsity(brain(v - sm, m))
        split["%-6s sigma=%2d" % (name, s)] = dict(smooth_std=float(np.std(brain(sm, m))),
                                                  rem_std=st["std"], kurtosis=st["kurtosis"],
                                                  top1=st["top1"], top5=st["top5"])
table(split, title="value = smooth field (Gaussian, sigma px) + remainder")
concentration({"D_fit": df, "log Q": brain(LQ, VALID),
               "D_fit - smooth(%d)" % S_SHOW: brain(D_fit - mblur(D_fit, M, S_SHOW)),
               "log Q - smooth(%d)" % S_SHOW: brain(LQ_rem, VALID),
               "Laplace": ref.laplace(size=200000)},
              "difference vs log-quotient: which is sparser?")

## 3. Transfer function

Everything here is for the operator's direction, **CT1 -> T1** (`x = CT1`, `y = T1`).

### 3a. Spectra: is the difference high-frequency?

If enhancement lived at high `|k|`, the spectrum of `D` would rise relative to T1's. A sparse,
localised `D` instead has a flatter spectrum than anatomy at every frequency. The table gives the
share of each image's energy per `kfilter` band.

In [ ]:
def rfreq(Hh, Ww):
    return np.sqrt(np.fft.fftfreq(Hh)[:, None] ** 2 + np.fft.fftfreq(Ww)[None, :] ** 2)


def radial_profile(S, nb=NRAD):
    """Mean of a full-grid (H, W) spectrum in nb radial bins over [0, 0.5] cycles / pixel."""
    S = np.asarray(S)
    if np.iscomplexobj(S):
        return radial_profile(S.real, nb) + 1j * radial_profile(S.imag, nb)
    idx = np.minimum((rfreq(*S.shape) / 0.5 * nb).astype(int), nb).ravel()   # nb = the corners
    cnt = np.bincount(idx, minlength=nb + 1)
    return (np.bincount(idx, weights=S.ravel(), minlength=nb + 1) / np.maximum(cnt, 1))[:nb]


RC = (np.arange(NRAD) + 0.5) * 0.5 / NRAD                        # bin centres, cycles / pixel
NODES = [0.5 * 2.0 ** (j - (BANDS - 1)) for j in range(1, BANDS)]  # kfilter band nodes

X, Y = torch.fft.fft2(CT1), torch.fft.fft2(T1)
Sxx = (X.abs() ** 2).mean(0).cpu().numpy()
Syy = (Y.abs() ** 2).mean(0).cpu().numpy()
Sdd = ((X - Y).abs() ** 2).mean(0).cpu().numpy()
Sxy = (X.conj() * Y).mean(0).cpu().numpy()

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for label, S in (("T1", Syy), ("CT1", Sxx), ("D = CT1 - T1", Sdd)):
    ax.loglog(RC, radial_profile(S), label=label)
for nd in NODES:
    ax.axvline(nd, color="k", lw=0.4, alpha=0.4)
ax.set(xlabel="|k| (cycles / pixel)", ylabel="mean power", title="radial power spectra (lines: band nodes)")
ax.grid(alpha=0.3, which="both"); ax.legend(); plt.show()


def band_shares(S_full, n=BANDS):
    """Share of total power in each band (the bands are a partition of unity, so shares sum to 1)."""
    Hh, Ww = S_full.shape
    beta = radial_bands(Hh, Ww, n, dtype=torch.float64).numpy()
    w = np.full(Ww // 2 + 1, 2.0); w[0] = 1.0
    if Ww % 2 == 0:
        w[-1] = 1.0
    e = (beta * (S_full[:, :Ww // 2 + 1] * w)[None]).sum((1, 2))
    return e / e.sum()


BAND_LBL = ["<%.3g" % NODES[0]] + ["~%.3g" % nd for nd in NODES]   # band j peaks at node j
shares = {nm: dict(zip(BAND_LBL, band_shares(S))) for nm, S in
          (("T1", Syy), ("CT1", Sxx), ("D", Sdd))}
shares["D / T1"] = {k: shares["D"][k] / max(shares["T1"][k], 1e-30) for k in BAND_LBL}
table(shares, fmt="%10.3g", title="share of energy per kfilter band (band peak, cycles / pixel)")

### 3b. The global transfer function and coherence

`W(k) = S_xy / S_xx` is the least-squares shift-invariant filter CT1 -> T1, and
`coherence = |S_xy|^2 / (S_xx S_yy)` is the fraction of T1 that ANY linear filter can explain at
that frequency. Both are averaged over the pooled slices.

* `Re W` near 1 with coherence near 1: T1 and CT1 agree there, the band is pinned.
* coherence falling with `|k|`: unexplained -- enhancement, but also independent noise and
  sub-pixel misalignment, which all look the same in this plot.
* `Im W` away from 0, or a ramp in the 2-D phase map, is a **shift** between T1 and CT1.

The 2-D maps check the radial assumption: a strongly non-circular pattern means radial bands are
the wrong parameterisation.

In [ ]:
Wr = radial_profile(Sxy) / radial_profile(Sxx)
coh = np.abs(radial_profile(Sxy)) ** 2 / (radial_profile(Sxx) * radial_profile(Syy))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].semilogx(RC, Wr.real, label="Re W"); axs[0].semilogx(RC, Wr.imag, label="Im W")
axs[0].axhline(1, color="k", lw=0.5); axs[0].axhline(0, color="k", lw=0.5)
axs[0].set(xlabel="|k| (cycles / pixel)", title="global transfer function CT1 -> T1")
axs[1].semilogx(RC, coh, label="coherence")
axs[1].semilogx(RC, 1 - coh, label="1 - coherence (unexplained)")
axs[1].set(xlabel="|k| (cycles / pixel)", ylim=(0, 1.02), title="coherence (N = %d slices)" % N)
for a in axs:
    for nd in NODES:
        a.axvline(nd, color="k", lw=0.4, alpha=0.4)
    a.grid(alpha=0.3, which="both"); a.legend()
plt.show()

sh = np.fft.fftshift
W2 = Sxy / np.maximum(Sxx, 1e-30)
coh2 = np.abs(Sxy) ** 2 / np.maximum(Sxx * Syy, 1e-60)
subplot_images([sh(W2.real), sh(np.angle(Sxy)), sh(coh2), sh(np.log10(Sxx + 1e-30))],
               titles=["Re W(k)", "phase of S_xy (rad)", "coherence", "log10 power of CT1"],
               cmap=["bwr", "twilight", "viridis", "magma"],
               vmin=[0.0, -np.pi, 0.0, None], vmax=[2.0, np.pi, 1.0, None],
               share_window=False, orient=None, colorbar="each", panel_size=(3.4, 3.6),
               suptitle="2-D k-space maps (DC at the centre)")

### 3c. Band gains: one transfer function for everything, or one per image?

The `kfilter` model with scalar gains, `T1 ~ sum_j a_j * (B_j CT1)`, solved by least squares --
once on the pooled slices and once per slice. The spread across slices is the prior on a *global*
transfer function: a tight band means one learned filter is enough; a wide one means the gains
have to be predicted per image (or per pixel, 3d).

In [ ]:
def split(x, n):
    """(N, H, W) -> (N, n, H, W): x filtered into the n radial bands (they sum to x)."""
    beta = radial_bands(x.shape[-2], x.shape[-1], n, x.device, x.dtype)
    return torch.fft.irfft2(torch.fft.rfft2(x)[:, None] * beta, s=x.shape[-2:])


def rmse_brain(pred, target=None, m=None):
    target = T1 if target is None else target
    m = M if m is None else m
    return float(((pred - target)[m] ** 2).mean().sqrt())


Xb = split(CT1, BANDS)
G = torch.einsum("nihw,njhw->nij", Xb, Xb).double().cpu().numpy()
r = torch.einsum("nihw,nhw->ni", Xb, T1).double().cpu().numpy()
eye = np.eye(BANDS)
ridge = 1e-6 * np.trace(G, axis1=1, axis2=2)[:, None, None] / BANDS * eye
a_slice = np.linalg.solve(G + ridge, (r + ridge.sum(-1))[..., None])[..., 0]      # (N, BANDS)
a_pool = np.linalg.solve(G.sum(0), r.sum(0))

lo, med, hi = np.percentile(a_slice, [10, 50, 90], axis=0)
fig, ax = plt.subplots(figsize=(7, 4))
xs = np.arange(BANDS)
ax.fill_between(xs, lo, hi, alpha=0.25, label="per slice, 10-90%")
ax.plot(xs, med, "o-", label="per slice, median")
ax.plot(xs, a_pool, "s--", label="pooled fit")
ax.axhline(1, color="k", lw=0.5)
ax.set_xticks(xs); ax.set_xticklabels(BAND_LBL, rotation=30)
ax.set(xlabel="band peak (cycles / pixel)", ylabel="gain a_j", title="scalar band gains, CT1 -> T1")
ax.grid(alpha=0.3); ax.legend(); plt.show()
table({"pooled": dict(zip(BAND_LBL, a_pool)), "slice median": dict(zip(BAND_LBL, med)),
       "slice p10": dict(zip(BAND_LBL, lo)), "slice p90": dict(zip(BAND_LBL, hi))}, fmt="%10.3f")

g_e, c_e, _ = affine_fit(CT1, T1, M)                       # the operator's direction: T1 ~ g*CT1 + c
ta = lambda a: torch.as_tensor(a, dtype=T1.dtype, device=T1.device)
FIT = {"identity (T1 ~ CT1)": rmse_brain(CT1),
       "per-slice gain + offset": rmse_brain(g_e[:, None, None] * CT1 + c_e[:, None, None]),
       "pooled %d-band filter" % BANDS: rmse_brain((ta(a_pool)[None, :, None, None] * Xb).sum(1)),
       "per-slice %d-band filter" % BANDS: rmse_brain((ta(a_slice)[:, :, None, None] * Xb).sum(1))}
print("\nrmse of T1 inside the brain (in-sample):")
for k, v in FIT.items():
    print("  %-28s %.5f   (%.3fx identity)" % (k, v, v / FIT["identity (T1 ~ CT1)"]))
del Xb

### 3d. Per-pixel gain maps: what the UNet would have to predict

The same model with gains that vary over the image, fitted by least squares in a Gaussian window
of `LOCAL_SIGMA` px around every pixel (ridged toward 1). These maps are the target of the
`kfilter` UNet -- an **oracle**, since the fit sees T1.

**Look for:** are the maps `1` almost everywhere with compact departures (sparse about 1)? Smooth?
Do the bands disagree, or is it one map repeated -- in which case a single `gain` map is enough?

In [ ]:
def local_gains(Xb, y, sigma, lam=RIDGE):
    """Per-pixel ridge least squares for y ~ sum_j a_j(p) * Xb_j(p) over a Gaussian window.
    Xb (n, J, H, W), y (n, H, W) -> a (n, J, H, W). numpy solve: local torch has no LAPACK."""
    n, J, Hh, Ww = Xb.shape
    Gp = blur((Xb[:, :, None] * Xb[:, None]).reshape(n, J * J, Hh, Ww), sigma)
    rp = blur(Xb * y[:, None], sigma)
    Gp = Gp.reshape(n, J, J, Hh, Ww).permute(0, 3, 4, 1, 2).double().cpu().numpy()
    rp = rp.permute(0, 2, 3, 1).double().cpu().numpy()
    reg = lam * np.trace(Gp, axis1=-2, axis2=-1) / J + 1e-12           # (n, H, W)
    A = Gp + reg[..., None, None] * np.eye(J)
    a = np.linalg.solve(A, (rp + reg[..., None])[..., None])[..., 0]
    return torch.as_tensor(a, dtype=y.dtype, device=y.device).permute(0, 3, 1, 2)


NL = min(N_LOCAL, N)
sl = slice(0, NL)
XbJ = split(CT1[sl], J_LOCAL)
A_loc = local_gains(XbJ, T1[sl], LOCAL_SIGMA)                          # (NL, J_LOCAL, H, W)
A_one = local_gains(CT1[sl, None], T1[sl], LOCAL_SIGMA)[:, 0]          # the single-map `gain` oracle
LNODES = [0.5 * 2.0 ** (j - (J_LOCAL - 1)) for j in range(1, J_LOCAL)]
LBL = ["<%.3g" % LNODES[0]] + ["~%.3g" % nd for nd in LNODES]

RL = [i for i in R if i < NL]
A_WIN = 0.5
subplot_images([[T1[i], CT1[i], (A_one[i] - 1) * M[i]] + [(A_loc[i, j] - 1) * M[i] for j in range(J_LOCAL)]
                for i in RL],
               col_titles=["T1", "CT1", "single gain"] + ["band %s" % l for l in LBL],
               panel_size=(2.8, 2.9),
               cmap=["gray", "gray"] + ["bwr"] * (J_LOCAL + 1),
               vmin=[0.0, 0.0] + [-A_WIN] * (J_LOCAL + 1), vmax=[V_IMG, V_IMG] + [A_WIN] * (J_LOCAL + 1),
               row_labels=["#%d" % i for i in RL], colorbar=True, cbar_label="a - 1",
               suptitle="local gain maps a - 1, CT1 -> T1, window sigma = %d px (white = gain of 1)"
                        % LOCAL_SIGMA)

Ml = M[sl]
subplot_hists({"single gain": brain(A_one, Ml), **{"band %s" % l: brain(A_loc[:, j], Ml)
                                                    for j, l in enumerate(LBL)}},
              bins=200, range=(1 - 3 * A_WIN, 1 + 3 * A_WIN), log=True, vlines={"gain = 1": 1.0}, xlabel="gain",
              ncols=min(J_LOCAL + 1, 5), suptitle="distribution of the local gains inside the brain")
table({"single gain": sparsity(brain(A_one, Ml)),
       **{"band %s" % l: sparsity(brain(A_loc[:, j], Ml)) for j, l in enumerate(LBL)}, **REFS},
      title="\nconcentration of (gain - median gain) inside the brain")

cc = np.corrcoef(np.stack([brain(A_loc[:, j], Ml) for j in range(J_LOCAL)]))
print("\ncorrelation between band gain maps (near 1 everywhere = one map would do):")
print(np.array2string(cc, precision=2, suppress_small=True))

### 3e. How smooth can the gain maps be and still explain T1?

The window `sigma` IS the prior: with no smoothing a single gain map fits T1 exactly
(`a = T1 / CT1`, section 2) and says nothing. So the useful curve is fit against smoothness. If the
banded curve sits well below the single-gain curve at large `sigma`, splitting by frequency lets
smooth maps do the work; if they coincide, `kfilter` buys nothing over `gain`.

In [ ]:
curve = {"single gain": [], "%d bands" % J_LOCAL: []}
for s in SIGMAS:
    a1 = local_gains(CT1[sl, None], T1[sl], s)[:, 0]
    curve["single gain"].append(rmse_brain(a1 * CT1[sl], T1[sl], Ml))
    aJ = local_gains(XbJ, T1[sl], s)
    curve["%d bands" % J_LOCAL].append(rmse_brain((aJ * XbJ).sum(1), T1[sl], Ml))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for k, v in curve.items():
    ax.plot(SIGMAS, v, "o-", label="local " + k)
for (k, v), ls in zip(FIT.items(), ("-", "--", "-.", ":")):
    ax.axhline(v, color="gray", ls=ls, lw=1, label=k)
ax.set_xscale("log", base=2)
ax.set(xlabel="window sigma (px) -- smoother maps to the right", ylabel="rmse of T1 in the brain",
       title="oracle fit vs smoothness of the gain maps")
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8); plt.show()
table({k: dict(zip(["s=%d" % s for s in SIGMAS], v)) for k, v in curve.items()}, fmt="%10.5f")

## Redraw

`draw(n, seed)` pools a different set of slices; re-run the sections afterwards. `SHOW = [..]`
picks which pooled slices the image panels display.